In [1]:
!pip install anthropic rank_bm25 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 15.2 MB/s eta 0:00:00


In [2]:
import anthropic
from google.colab import userdata
from rank_bm25 import BM25Okapi

client = anthropic.Anthropic(api_key=userdata.get('dinkey'))

In [3]:
document = """
VACATION POLICY
Employees are entitled to 20 days of paid vacation per year.
Vacation days must be approved by your manager at least 2 weeks in advance.
Unused vacation days can be carried over to the next year, up to a maximum of 10 days.

SICK LEAVE POLICY
Employees receive 10 sick days per year.
Sick days do not carry over to the next year.
A doctor's note is required for sick leave longer than 3 consecutive days.

REMOTE WORK POLICY
Employees may work remotely up to 3 days per week.
Remote work must be approved by your manager.
Employees must be available during core hours of 10am to 3pm.

EXPENSE POLICY
Expenses under $50 can be reimbursed without prior approval.
Expenses over $50 require manager approval before being incurred.
All expenses must be submitted within 30 days with receipts.
"""

chunks = [chunk.strip() for chunk in document.split("\n\n") if chunk.strip()]
tokenized_chunks = [chunk.lower().split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)
print(f"BM25 ready with {len(chunks)} chunks")

BM25 ready with 4 chunks


In [4]:
tools1 = [
    {
        "name": "calculate",
        "description": "Performs math. Use when user asks to calculate something.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Math expression e.g. '2 + 2'"}
            },
            "required": ["expression"]
        }
    },
    {
        "name": "search_document",
        "description": "Searches company policy document. Use for questions about vacation, sick leave, remote work, or expenses.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The question to search for"}
            },
            "required": ["query"]
        }
    }
]

def calculate(expression):
    try:
        return str(eval(expression))
    except Exception as e:
        return str(e)

def search_document(query):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:2]
    results = [chunks[i] for i in top_indices]
    return "\n\n".join(results)

# maps tool name → function
tool_map = {
    "calculate": calculate,
    "search_document": search_document
}

In [5]:
conversation = []

def chat(user_message):
    conversation.append({"role": "user", "content": user_message})

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        tools=tools1,
        messages=conversation
    )

    if response.stop_reason == "tool_use":
        tool_use = response.content[0]

        # dynamically call the right function with the right arguments
        result = tool_map[tool_use.name](**tool_use.input)
        print(f"(called '{tool_use.name}' with {tool_use.input})")

        conversation.append({"role": "assistant", "content": response.content})
        conversation.append({"role": "user", "content": [
            {"type": "tool_result", "tool_use_id": tool_use.id, "content": result}
        ]})

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=512,
            tools=tools1,
            messages=conversation
        )

    reply = response.content[0].text
    conversation.append({"role": "assistant", "content": reply})
    print(f"Claude: {reply}\n")

In [6]:
conversation = []  # fresh start

chat("how many vacation days do I get?")       # should use search_document
chat("what is 347 * 28?")                      # should use calculate
chat("can I work from home every day?")        # should use search_document
chat("multiply the vacation days by 3")        # should use calculate, and remember 20 days
chat("what is the capital of France?")         # no tool needed

(called 'search_document' with {'query': 'vacation days'})
Claude: According to the company policy, you are entitled to **20 days of paid vacation per year**.

A few important details:
- Vacation days must be approved by your manager at least 2 weeks in advance
- You can carry over up to 10 unused vacation days to the next year

Is there anything else you'd like to know about vacation or other time-off policies?

(called 'calculate' with {'expression': '347 * 28'})
Claude: 347 * 28 = **9,716**

(called 'search_document' with {'query': 'remote work from home'})
Claude: No, according to the remote work policy, you can work from home **up to 3 days per week** (not every day).

Here are the key requirements:
- Remote work must be approved by your manager
- You must be available during core hours from 10am to 3pm

So you can work remotely a maximum of 3 days per week, with the other 2 days needing to be in the office (assuming a standard 5-day work week).

(called 'calculate' with {'express